In [21]:
import sys
from pathlib import Path

project_root = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "src" / "airport_encoding.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

flight = pd.read_csv(project_root / "data" / "flights.csv")

## Data preview and basic information
Each row represents a flight. The previews below retain the original data; negative delays indicate early departures or arrivals.

In [22]:
from IPython.display import display

# First 10 rows, with every column visible
with pd.option_context("display.max_columns", None):
    display(flight.head(10))

,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,name
0,0,2013,1,1,517.0,515,2.0,830.0,819,11.0,UA,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,United Air Lines Inc.
1,1,2013,1,1,533.0,529,4.0,850.0,830,20.0,UA,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00,United Air Lines Inc.
2,2,2013,1,1,542.0,540,2.0,923.0,850,33.0,AA,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00,American Airlines Inc.
3,3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,B6,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00,JetBlue Airways
4,4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,DL,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01 06:00:00,Delta Air Lines Inc.
5,5,2013,1,1,554.0,558,-4.0,740.0,728,12.0,UA,1696,N39463,EWR,ORD,150.0,719,5,58,2013-01-01 05:00:00,United Air Lines Inc.
6,6,2013,1,1,555.0,600,-5.0,913.0,854,19.0,B6,507,N516JB,EWR,FLL,158.0,1065,6,0,2013-01-01 06:00:00,JetBlue Airways
7,7,2013,1,1,557.0,600,-3.0,709.0,723,-14.0,EV,5708,N829AS,LGA,IAD,53.0,229,6,0,2013-01-01 06:00:00,ExpressJet Airlines Inc.
8,8,2013,1,1,557.0,600,-3.0,838.0,846,-8.0,B6,79,N593JB,JFK,MCO,140.0,944,6,0,2013-01-01 06:00:00,JetBlue Airways
9,9,2013,1,1,558.0,600,-2.0,753.0,745,8.0,AA,301,N3ALAA,LGA,ORD,138.0,733,6,0,2013-01-01 06:00:00,American Airlines Inc.


In [23]:
# A reproducible sample from across the dataset
with pd.option_context("display.max_columns", None):
    display(flight.sample(10, random_state=42))

,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,name
159280,159280,2013,3,25,1929.0,1905,24.0,2236.0,2217,19.0,UA,1471,N37298,EWR,RSW,169.0,1068,19,5,2013-03-25 19:00:00,United Air Lines Inc.
189101,189101,2013,4,26,956.0,1000,-4.0,1257.0,1334,-37.0,DL,1765,N717TW,JFK,SFO,337.0,2586,10,0,2013-04-26 10:00:00,Delta Air Lines Inc.
212435,212435,2013,5,21,1320.0,1309,11.0,1430.0,1414,16.0,EV,4129,N11536,EWR,DCA,39.0,199,13,9,2013-05-21 13:00:00,ExpressJet Airlines Inc.
266804,266804,2013,7,18,1222.0,1230,-8.0,1357.0,1419,-22.0,EV,5796,N13958,EWR,CLT,77.0,529,12,30,2013-07-18 12:00:00,ExpressJet Airlines Inc.
306581,306581,2013,8,29,540.0,545,-5.0,921.0,921,0.0,B6,939,N535JB,JFK,BQN,198.0,1576,5,45,2013-08-29 05:00:00,JetBlue Airways
68828,68828,2013,11,14,2031.0,2025,6.0,2342.0,2340,2.0,UA,314,N518UA,JFK,LAX,342.0,2475,20,25,2013-11-14 20:00:00,United Air Lines Inc.
164026,164026,2013,3,30,1755.0,1755,0.0,2110.0,2114,-4.0,B6,675,N646JB,JFK,LAX,338.0,2475,17,55,2013-03-30 17:00:00,JetBlue Airways
289110,289110,2013,8,10,1544.0,1545,-1.0,2003.0,2001,2.0,DL,4,N3764D,JFK,SJU,205.0,1598,15,45,2013-08-10 15:00:00,Delta Air Lines Inc.
161286,161286,2013,3,27,1855.0,1900,-5.0,2019.0,2028,-9.0,9E,4091,N8623A,JFK,BWI,37.0,184,19,0,2013-03-27 19:00:00,Endeavor Air Inc.
205307,205307,2013,5,13,1912.0,1915,-3.0,2055.0,2154,-59.0,DL,245,N3736C,JFK,PHX,262.0,2153,19,15,2013-05-13 19:00:00,Delta Air Lines Inc.


In [24]:
dates = pd.to_datetime(flight[["year", "month", "day"]])
print(f"Rows: {flight.shape[0]:,} | Columns: {flight.shape[1]}")
print(f"Date range: {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d}")
print(f"Origins: {flight.origin.nunique()} | Destinations: {flight.dest.nunique()} | Airlines: {flight.carrier.nunique()}")
print(f"Duplicate records excluding id: {flight.drop(columns="id").duplicated().sum():,}")
print()
flight.info()

Rows: 336,776 | Columns: 21
Date range: 2013-01-01 to 2013-12-31
Origins: 3 | Destinations: 105 | Airlines: 16
Duplicate records excluding id: 0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336776 entries, 0 to 336775
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              336776 non-null  int64  
 1   year            336776 non-null  int64  
 2   month           336776 non-null  int64  
 3   day             336776 non-null  int64  
 4   dep_time        328521 non-null  float64
 5   sched_dep_time  336776 non-null  int64  
 6   dep_delay       328521 non-null  float64
 7   arr_time        328063 non-null  float64
 8   sched_arr_time  336776 non-null  int64  
 9   arr_delay       327346 non-null  float64
 10  carrier         336776 non-null  object 
 11  flight          336776 non-null  int64  
 12  tailnum         334264 non-null  object 
 13  origin          336776 non-null  object 
 14  de

In [25]:
# Data types, missing values, and number of distinct values
column_summary = pd.DataFrame({
    "dtype": flight.dtypes.astype(str),
    "non_null": flight.notna().sum(),
    "missing": flight.isna().sum(),
    "missing_pct": flight.isna().mean().mul(100).round(2),
    "unique_values": flight.nunique()
})
display(column_summary)

,dtype,non_null,missing,missing_pct,unique_values
id,int64,336776,0,0.00,336776
year,int64,336776,0,0.00,1
month,int64,336776,0,0.00,12
day,int64,336776,0,0.00,31
dep_time,float64,328521,8255,2.45,1318
sched_dep_time,int64,336776,0,0.00,1021
dep_delay,float64,328521,8255,2.45,527
arr_time,float64,328063,8713,2.59,1411
sched_arr_time,int64,336776,0,0.00,1163
arr_delay,float64,327346,9430,2.80,577


In [26]:
# Summary statistics (missing values excluded)
display(flight[["dep_delay", "arr_delay", "air_time", "distance"]].describe().round(2))

,dep_delay,arr_delay,air_time,distance
count,328521.00,327346.00,327346.00,336776.00
mean,12.64,6.90,150.69,1039.91
std,40.21,44.63,93.69,733.23
min,-43.00,-86.00,20.00,17.00
25%,-5.00,-17.00,82.00,502.00
50%,-2.00,-5.00,129.00,872.00
75%,11.00,14.00,192.00,1389.00
max,1301.00,1272.00,695.00,4983.00


### Initial observations
- `origin` and `dest` are categorical location features suitable for the encoding comparison.
- `time_hour` loads as text and should be parsed as a datetime when needed.
- `id` is an identifier, rather than a meaningful numeric predictor.
- Delay and air-time fields contain missing values; account for these before modeling.
- The delay means exceed the medians, indicating that large delays pull the averages upward.

## Data cleaning issues and handling

The source dataset contains **336,776 flights and 21 columns**. The following checks were performed on `data/flights.csv` before documenting the encoding workflow.

### Missing values

| Column | Missing rows | Percentage of all flights |
| --- | ---: | ---: |
| `dep_time` | 8,255 | 2.45% |
| `dep_delay` | 8,255 | 2.45% |
| `arr_time` | 8,713 | 2.59% |
| `arr_delay` | 9,430 | 2.80% |
| `tailnum` | 2,512 | 0.75% |
| `air_time` | 9,430 | 2.80% |

These counts overlap; they should not be added together. All other columns have no missing values.

**Handling already implemented:** `arr_delay` is the prediction target. The encoding code below uses `flight.dropna(subset=[TARGET_COLUMN])` before splitting the data. This excludes the **9,430 rows without an observed target**, leaving **327,346 flights** for the shared training/test split. In this dataset, those retained rows have **no missing values in any column**, so no additional missing-value imputation is needed for this subset. The original `flight` DataFrame remains unchanged. Missing arrival delays should not be replaced with zero, because zero means an observed on-time arrival. Results describe flights with recorded arrival delays; the reason each excluded value is missing has not been established from these checks.

### Time formats and predictor preparation

- `time_hour` loads as text. All values parse successfully as datetimes, but convert it with `pd.to_datetime` if using datetime features.
- `dep_time`, `sched_dep_time`, `arr_time`, and `sched_arr_time` use HHMM notation rather than elapsed minutes. There are **29 departure times and 150 arrival times recorded as `2400`**. Treat these as midnight and account for the date rollover when constructing timestamps; do not treat HHMM values as ordinary continuous measurements. No other invalid HHMM values were found among nonmissing entries.
- `id` is preserved for matching flights across datasets. Exclude it when selecting model predictors.

### Negative and extreme delays

Negative delays indicate early departures or arrivals and should be retained. Observed departure delays range from **-43 to 1,301 minutes**, and arrival delays range from **-86 to 1,272 minutes**. The large positive values warrant inspection for their influence on model performance, but their size alone does not establish that they are errors. The current code does not trim, cap, or remove these values.

### Checks that found no cleaning problems

- **No duplicate flight records** were found after excluding `id` from the comparison.
- `origin` and `dest` have **no missing values**, and every airport code is covered by the local coordinate lookup.
- No blank strings or leading/trailing whitespace were found in the loaded text columns.
- All year/month/day combinations and `time_hour` strings parse as valid dates or timestamps.
- Nonmissing `air_time` and `distance` values are positive.

The workflow excludes flights with missing arrival delay, then builds five airport representations with identical one-hot features for carrier, month, and weekday.


## Encoding `origin` and `dest`

We compare five representations of the two airport-code columns. Each variant uses
only its airport features to predict arrival delay. Original row indices are preserved.

| Strategy | Representation | Main tradeoff |
| --- | --- | --- |
| **One-hot** | One 0/1 column per airport, separately for origin and destination. | Preserves identity without order, but adds columns. |
| **Ordinal** | An integer ID for each airport, assigned alphabetically. | Compact, but the numeric order and distances are arbitrary. |
| **Frequency** | The airport's proportion of rows in the training set. | Compact; airports with equal frequency become indistinguishable. |
| **Smoothed target** | An airport's average outcome, shrunk toward the training mean. | Uses predictive information, but requires a target and careful fitting to avoid leakage. |
| **Geographic** | Latitude and longitude for each airport (four columns total). | Captures physical location, but cannot directly represent every airport-specific effect. |

The same 80/20 random split is used for every strategy. Category vocabularies and
frequencies are learned only from the training rows. Target encoding uses internal
five-fold cross-fitting for training rows; test rows use mappings learned from the
training set only. Coordinates come from a fixed external lookup, not outcomes.

This is a reusable holdout comparison, not a future-flight forecasting split. For
cross-validation, call `make_encoded_datasets` separately inside **each** fold on
the raw fold data; do not cross-validate precomputed target-encoded training data.

References: [scikit-learn TargetEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html),
[nycflights13 airport data](https://github.com/tidyverse/nycflights13/blob/main/data-raw/airports.csv),
and [airport-codes / OurAirports](https://github.com/datasets/airport-codes).

In [27]:
# Requires scikit-learn >= 1.3 for TargetEncoder.
from src.airport_encoding import (
    AIRPORT_COLUMNS,
    RANDOM_STATE,
    TARGET_COLUMN,
    make_encoded_datasets,
    sample_flights,
)

airport_coordinates = pd.read_csv(project_root / "data" / "airport_coordinates.csv").set_index("airport")
assert airport_coordinates.index.is_unique
assert airport_coordinates[["latitude", "longitude"]].notna().all().all()

### Reusable encoding function

`src/airport_encoding.py` provides the shared sampler and encoders. Each result contains only the target and encoded `origin`/`dest` features. Learned
encodings use training rows only. Unseen airports become all zeros for one-hot,
`-1` for ordinal, `0` for frequency, and the training target mean for target
encoding. Geographic encoding uses the fixed coordinate lookup and raises an error
if airport coordinates are missing.

For smoothed target encoding we use a prior weight of 20:
`(airport_count * airport_mean + 20 * global_mean) / (airport_count + 20)`.
Training values use cross-fitting so a row's own target does not contribute to its
encoded feature.


### Create the modified datasets

Rows missing `TARGET_COLUMN` are excluded. One random sample of 100,000 rows is
split 80/20 and shared by all five encodings.

Each dataset contains only the selected encoding of `origin` and `dest`, plus the
target column. Access a dataset with
`encoded_datasets["one_hot"]["train"]`, replacing `one_hot` with `ordinal`,
`frequency`, `smoothed_target`, or `geographic`. Use the encoded airport feature
columns as predictors and `TARGET_COLUMN` as y. No other flight columns are
included in these modeling datasets.


In [29]:
if TARGET_COLUMN is not None:
    print(f"Rows excluded for missing {TARGET_COLUMN}: {flight[TARGET_COLUMN].isna().sum():,}")

raw_train, raw_test = sample_flights(flight, TARGET_COLUMN)
print(f"Shared random sample: {len(raw_train) + len(raw_test):,} flights (seed={RANDOM_STATE}).")
encoded_datasets, fitted_encoders, encoded_columns = make_encoded_datasets(
    raw_train, raw_test, airport_coordinates, target_column=TARGET_COLUMN
)

encoding_summary = pd.DataFrame([
    {"strategy": name,
     "train_rows": len(parts["train"]),
     "test_rows": len(parts["test"]),
     "airport_features": len(encoded_columns[name]),
     "total_columns": parts["train"].shape[1]}
    for name, parts in encoded_datasets.items()
]).set_index("strategy")
display(encoding_summary)

# Confirm every dataset contains only its airport features and target.
for name, parts in encoded_datasets.items():
    expected_columns = set(encoded_columns[name])
    assert TARGET_COLUMN not in expected_columns
    if TARGET_COLUMN is not None:
        expected_columns.add(TARGET_COLUMN)
    for split, original in [("train", raw_train), ("test", raw_test)]:
        result = parts[split]
        assert result.index.equals(original.index)
        assert set(result.columns) == expected_columns
        if TARGET_COLUMN is not None:
            assert result[TARGET_COLUMN].equals(original[TARGET_COLUMN])
        assert result[encoded_columns[name]].notna().all().all()
print("Verified: datasets contain only encoded origin/dest features and target.")


Rows excluded for missing arr_delay: 9,430
Shared random sample: 100,000 flights (seed=42).


,train_rows,test_rows,airport_features,total_columns
strategy,,,,
one_hot,80000,20000,106,107
ordinal,80000,20000,2,3
frequency,80000,20000,2,3
smoothed_target,80000,20000,2,3
geographic,80000,20000,4,5


Verified: datasets contain only encoded origin/dest features and target.


In [30]:
# Compare every column for the same five flights before and after encoding.
preview_index = raw_train.index[:5]
with pd.option_context("display.max_columns", None):
    display(raw_train.loc[preview_index])
    for strategy, parts in encoded_datasets.items():
        print(strategy)
        display(parts["train"].loc[preview_index])

# Later, reuse the same datasets for each model configuration, for example:
# train_one_hot = encoded_datasets["one_hot"]["train"]
# X_train = train_one_hot.drop(columns=[TARGET_COLUMN])
# y_train = train_one_hot[TARGET_COLUMN]
# Select/preprocess the remaining predictors before model fitting.

,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,name
270063,270063,2013,7,21,2144.0,2059,45.0,2352.0,2307,45.0,EV,5254,N759EV,LGA,DSM,144.0,1031,20,59,2013-07-21 20:00:00,ExpressJet Airlines Inc.
188189,188189,2013,4,25,1129.0,1130,-1.0,1442.0,1354,48.0,EV,5216,N758EV,LGA,MCI,168.0,1107,11,30,2013-04-25 11:00:00,ExpressJet Airlines Inc.
116114,116114,2013,2,6,1557.0,1600,-3.0,1812.0,1820,-8.0,MQ,3985,N604MQ,JFK,CVG,108.0,589,16,0,2013-02-06 16:00:00,Envoy Air
242815,242815,2013,6,22,2244.0,2245,-1.0,1.0,4,-3.0,B6,108,N187JB,JFK,PWM,47.0,273,22,45,2013-06-22 22:00:00,JetBlue Airways
152737,152737,2013,3,18,1806.0,1715,51.0,2137.0,2005,92.0,WN,1035,N7740A,EWR,HOU,219.0,1411,17,15,2013-03-18 17:00:00,Southwest Airlines Co.


one_hot


,arr_delay,origin_EWR,origin_JFK,origin_LGA,dest_ABQ,dest_ACK,dest_ALB,dest_ATL,dest_AUS,dest_AVL,dest_BDL,dest_BGR,dest_BHM,dest_BNA,dest_BOS,dest_BQN,dest_BTV,dest_BUF,dest_BUR,dest_BWI,dest_BZN,dest_CAE,dest_CAK,dest_CHO,dest_CHS,dest_CLE,dest_CLT,dest_CMH,dest_CRW,dest_CVG,dest_DAY,dest_DCA,dest_DEN,dest_DFW,dest_DSM,dest_DTW,dest_EGE,dest_EYW,dest_FLL,dest_GRR,dest_GSO,dest_GSP,dest_HDN,dest_HNL,dest_HOU,dest_IAD,dest_IAH,dest_ILM,dest_IND,dest_JAC,dest_JAX,dest_LAS,dest_LAX,dest_LEX,dest_LGB,dest_MCI,dest_MCO,dest_MDW,dest_MEM,dest_MHT,dest_MIA,dest_MKE,dest_MSN,dest_MSP,dest_MSY,dest_MTJ,dest_MVY,dest_MYR,dest_OAK,dest_OKC,dest_OMA,dest_ORD,dest_ORF,dest_PBI,dest_PDX,dest_PHL,dest_PHX,dest_PIT,dest_PSE,dest_PSP,dest_PVD,dest_PWM,dest_RDU,dest_RIC,dest_ROC,dest_RSW,dest_SAN,dest_SAT,dest_SAV,dest_SBN,dest_SDF,dest_SEA,dest_SFO,dest_SJC,dest_SJU,dest_SLC,dest_SMF,dest_SNA,dest_SRQ,dest_STL,dest_STT,dest_SYR,dest_TPA,dest_TUL,dest_TVC,dest_TYS,dest_XNA
270063,45.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
188189,48.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
116114,-8.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
242815,-3.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
152737,92.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


ordinal


,arr_delay,origin_ordinal,dest_ordinal
270063,45.0,2,30
188189,48.0,2,51
116114,-8.0,1,25
242815,-3.0,1,77
152737,92.0,0,40


frequency


,arr_delay,origin_frequency,dest_frequency
270063,45.0,0.308000,0.001600
188189,48.0,0.308000,0.005712
116114,-8.0,0.334738,0.011550
242815,-3.0,0.334738,0.006387
152737,92.0,0.357262,0.006550


smoothed_target


,arr_delay,origin_target_mean,dest_target_mean
270063,45.0,5.829684,21.321220
188189,48.0,5.857685,12.770983
116114,-8.0,5.683389,11.641575
242815,-3.0,6.015706,13.127133
152737,92.0,9.076041,8.815196


geographic


,arr_delay,origin_latitude,origin_longitude,dest_latitude,dest_longitude
270063,45.0,40.777245,-73.872608,41.533972,-93.663083
188189,48.0,40.777245,-73.872608,39.297606,-94.713905
116114,-8.0,40.639751,-73.778925,39.048836,-84.667822
242815,-3.0,40.639751,-73.778925,43.646161,-70.309281
152737,92.0,40.692500,-74.168667,29.645419,-95.278889
